# 05 Colab NIFTY50 Batch Pipeline

Use this notebook to batch through the full `data/raw/nifty50/` CSV corpus in Colab. It downloads or caches raw intraday Parquet files, builds features for every symbol, trains LightGBM plus LSTM/GRU, and writes per-symbol artifacts so runs do not overwrite each other.

For a quick smoke test, set `MAX_FILES = 1` in the download, feature, or train cells.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess
import sys

BASE = Path('/content/drive/MyDrive/trading_system')
REPO_DIR = BASE / 'TradingBot26'
DATA_DIR = BASE / 'data'
MODEL_DIR = BASE / 'models'
LOG_DIR = BASE / 'logs'
REPORT_DIR = BASE / 'reports'
ARTIFACT_DIR = BASE / 'artifacts'

for path in (DATA_DIR / 'raw', DATA_DIR / 'processed', MODEL_DIR, LOG_DIR, REPORT_DIR, ARTIFACT_DIR):
    path.mkdir(parents=True, exist_ok=True)

if not (REPO_DIR / 'pyproject.toml').exists():
    raise FileNotFoundError(f'Copy or clone this repo to {REPO_DIR} before running this notebook')

%cd $REPO_DIR
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)])

In [ ]:
import importlib
from dataclasses import fields, replace

import pandas as pd

repo_path = str(REPO_DIR)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
for module_name in ('config', 'data.loader'):
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()

import config as config_module
from config import MarketConfig, PathConfig
from data.loader import MarketDataLoader, standardize_ohlcv

market_fields = {field.name for field in fields(MarketConfig)}
required_fields = {'local_intraday_path', 'local_intraday_pattern'}
if not required_fields.issubset(market_fields):
    raise RuntimeError(
        'Colab is importing a stale TradingBot26 config.py from '
        f'{config_module.__file__}. Pull the latest repo commit, rerun the install cell, '
        'then use Runtime > Restart runtime before rerunning this notebook.'
    )
print(f'Using TradingBot26 config from: {config_module.__file__}')

paths = PathConfig(
    root=BASE,
    raw_data_dir=DATA_DIR / 'raw',
    processed_data_dir=DATA_DIR / 'processed',
    artifact_dir=ARTIFACT_DIR,
    model_artifact_dir=MODEL_DIR,
    report_dir=REPORT_DIR,
)

## 1. Cache raw NIFTY50 files

This cell walks the `nifty50` raw folder, converts every `*_5minute.csv` file into the expected per-symbol intraday Parquet cache, and writes daily context alongside it.

In [ ]:
NIFTY50_SOURCE_DIRS = [
    DATA_DIR / 'raw' / '18',
    DATA_DIR / 'raw' / 'nifty50',
    REPO_DIR / 'data' / 'raw' / '18',
    REPO_DIR / 'data' / 'raw' / 'nifty50',
]
NIFTY50_CSV_FILES = sorted(
    {
        path
        for source_dir in NIFTY50_SOURCE_DIRS
        if source_dir.exists()
        for path in source_dir.glob('*_5minute.csv')
        if path.is_file() and not path.name.startswith('.')
    }
)

if not NIFTY50_CSV_FILES:
    print('No NIFTY50 5-minute CSV files found under the Colab raw folders.')
else:
    FORCE_REFRESH = False
    MAX_FILES = 0  # set to 1 for a quick smoke test
    processed = []
    for csv_path in NIFTY50_CSV_FILES:
        base_name = csv_path.stem.removesuffix('_5minute')
        market = MarketConfig(
            symbol=base_name,
            ticker=base_name,
            series='IDX' if base_name.upper().startswith(('NIFTY', 'BANK')) else 'EQ',
            interval='5m',
            intraday_source='local-csv',
            daily_source='intraday-resample',
            local_intraday_path=csv_path,
            lookback_days=55,
        )
        loader = MarketDataLoader(paths, market)
        intraday_result = loader.download_intraday(force_refresh=FORCE_REFRESH, source='local-csv')
        daily_result = loader.download_daily_context(force_refresh=FORCE_REFRESH)
        print(
            f"{csv_path.name} -> intraday={intraday_result.rows} rows, daily={daily_result.rows} rows, "
            f"output={intraday_result.path.name}"
        )
        processed.append((csv_path.name, intraday_result.path, daily_result.path))
        if MAX_FILES and len(processed) >= MAX_FILES:
            break
    print(f'Completed raw download for {len(processed)} NIFTY50 files.')

## 2. Build features

This cell reads the cached intraday Parquet files, loads daily context, and runs the feature pipeline for every symbol.

In [ ]:
from config import FeatureConfig, LabelConfig, MarketConfig, NormalizerConfig, PathConfig
from data.loader import MarketDataLoader
from features.pipeline import FeatureEngineeringPipeline

RAW_INTRADAY_FILES = sorted(
    {
        path
        for path in (DATA_DIR / 'raw').glob('*_5m_intraday.parquet')
        if path.is_file() and not path.name.startswith('.')
    }
)

if not RAW_INTRADAY_FILES:
    print('No 5-minute intraday Parquet files found under data/raw.')
else:
    FORCE_REFRESH = False
    MAX_FILES = 0  # set to 1 for a quick smoke test
    processed = []
    labels = LabelConfig(horizon=15, target_pct=0.005, stop_pct=0.003)
    normalizer = NormalizerConfig(window=200, min_periods=200)
    features = FeatureConfig(lag_periods=(1, 5, 10), volume_confirm_window=20, include_daily_context=True)
    for intraday_path in RAW_INTRADAY_FILES:
        base_name = intraday_path.stem.removesuffix('_5m_intraday')
        market = MarketConfig(
            symbol=base_name,
            ticker=base_name,
            series='IDX' if base_name.upper().startswith(('NIFTY', 'BANK')) else 'EQ',
            interval='5m',
            intraday_source='cached-parquet',
            daily_source='intraday-resample',
        )
        loader = MarketDataLoader(paths, market)
        processed_path = FeatureEngineeringPipeline(
            paths=paths,
            market=market,
            features=features,
            labels=labels,
            normalizer=normalizer,
        ).processed_path()
        if processed_path.exists() and not FORCE_REFRESH:
            print(f'Skipping {intraday_path.name} because {processed_path.name} already exists.')
            processed.append((intraday_path.name, processed_path))
            if MAX_FILES and len(processed) >= MAX_FILES:
                break
            continue
        daily = loader.load_daily_context() if loader.daily_path().exists() else loader.download_daily_context(force_refresh=False)
        dataset = FeatureEngineeringPipeline(
            paths=paths,
            market=market,
            features=features,
            labels=labels,
            normalizer=normalizer,
        ).run(loader.load_intraday(), daily)
        print(
            f"{intraday_path.name} -> processed={len(dataset.frame):,} rows, features={len(dataset.feature_columns):,}, "
            f"output={dataset.feature_config_path.name}"
        )
        processed.append((intraday_path.name, dataset.feature_config_path))
        if MAX_FILES and len(processed) >= MAX_FILES:
            break
    print(f'Completed feature engineering for {len(processed)} files.')

## 3. Train models

This cell trains LightGBM, LSTM, and GRU on every processed feature file and writes per-symbol checkpoints and metadata.

In [ ]:
from pathlib import Path
import json
import torch

from config import FeatureConfig, LabelConfig, MarketConfig, NormalizerConfig, PathConfig, SplitConfig
from features.pipeline import FeatureEngineeringPipeline
from features.sequences import SequenceBuilder
from models.lgbm import LightGBMModel
from models.splits import chronological_split
from models.torch_models import GRUClassifier, LSTMClassifier
from models.torch_training import result_to_dict, train_torch_classifier, write_torch_training_metadata

LOOKBACK = 60
EPOCHS = 20
BATCH_SIZE = 512
LEARNING_RATE = 1e-3
NUM_WORKERS = 2
TRAIN_LIGHTGBM = True
TRAIN_LSTM = True
TRAIN_GRU = True
MAX_FILES = 0  # set to 1 for a quick smoke test

RAW_FEATURE_FILES = sorted(
    {
        path
        for path in (DATA_DIR / 'processed').glob('*_5m_features.parquet')
        if path.is_file() and not path.name.startswith('.')
    }
)

if not RAW_FEATURE_FILES:
    print('No 5-minute processed feature Parquet files found under data/processed.')
else:
    processed = []
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    effective_num_workers = NUM_WORKERS if device.type == 'cuda' else 0
    labels = LabelConfig()
    features = FeatureConfig()
    normalizer = NormalizerConfig()
    split_config = SplitConfig()
    for feature_path in RAW_FEATURE_FILES:
        base_name = feature_path.stem.removesuffix('_5m_features')
        market = MarketConfig(
            symbol=base_name,
            ticker=base_name,
            series='IDX' if base_name.upper().startswith(('NIFTY', 'BANK')) else 'EQ',
            interval='5m',
            intraday_source='processed-features',
            daily_source='intraday-resample',
        )
        pipeline = FeatureEngineeringPipeline(paths=paths, market=market, features=features, labels=labels, normalizer=normalizer)
        try:
            dataset = pipeline.load()
        except FileNotFoundError as exc:
            print(f'Skipping {feature_path.name}: {exc}')
            continue
        if dataset.frame.empty:
            print(f'Skipping {feature_path.name}: processed feature file is empty.')
            continue
        splits = chronological_split(dataset.frame, split_config)
        print(
            f"{feature_path.name} -> train={len(splits.train):,} val={len(splits.validation):,} test={len(splits.test):,} "
            f"features={len(dataset.feature_columns):,} device={device}"
        )
        lgbm_result = None
        if TRAIN_LIGHTGBM:
            lgbm_trainer = LightGBMModel(paths=paths, market=market)
            lgbm_result = lgbm_trainer.train(
                dataset.frame,
                feature_columns=dataset.feature_columns,
                split_config=split_config,
            )
            print(lgbm_result.model_path)
            print(lgbm_result.validation_metrics)
            print(lgbm_result.test_metrics)
        sequence_builder = SequenceBuilder(lookback=LOOKBACK)
        train_arrays = sequence_builder.build(splits.train, feature_columns=dataset.feature_columns)
        validation_arrays = sequence_builder.build(splits.validation, feature_columns=dataset.feature_columns)
        input_size = len(dataset.feature_columns)
        safe_name = base_name.replace('.', '_').replace('/', '_').replace(' ', '_')
        torch_results = {}
        if TRAIN_LSTM:
            lstm = LSTMClassifier(input_size=input_size, hidden_size_1=128, hidden_size_2=64, dropout=0.3)
            lstm_result = train_torch_classifier(
                model=lstm,
                model_name=f'lstm_{safe_name}_{market.interval}',
                train_arrays=train_arrays,
                validation_arrays=validation_arrays,
                model_dir=MODEL_DIR,
                epochs=EPOCHS,
                batch_size=BATCH_SIZE,
                learning_rate=LEARNING_RATE,
                num_workers=effective_num_workers,
                device=device,
            )
            torch_results['lstm'] = result_to_dict(lstm_result)
        if TRAIN_GRU:
            if device.type == 'cuda':
                torch.cuda.empty_cache()
            gru = GRUClassifier(input_size=input_size, hidden_size=128, dropout=0.3)
            gru_result = train_torch_classifier(
                model=gru,
                model_name=f'gru_{safe_name}_{market.interval}',
                train_arrays=train_arrays,
                validation_arrays=validation_arrays,
                model_dir=MODEL_DIR,
                epochs=EPOCHS,
                batch_size=BATCH_SIZE,
                learning_rate=LEARNING_RATE,
                num_workers=effective_num_workers,
                device=device,
            )
            torch_results['gru'] = result_to_dict(gru_result)
        metadata_path = MODEL_DIR / f'torch_training_{safe_name}_{market.interval}_metadata.json'
        write_torch_training_metadata(
            metadata_path,
            {
                'ticker': market.ticker,
                'symbol': market.symbol,
                'interval': market.interval,
                'lookback': LOOKBACK,
                'input_size': input_size,
                'feature_config': str(pipeline.feature_config_path()),
                'feature_columns': dataset.feature_columns,
                'device': str(device),
                'lgbm_model': str(lgbm_result.model_path) if lgbm_result else None,
                **torch_results,
            },
        )
        print(f'torch_metadata: {metadata_path}')
        processed.append(feature_path.name)
        if MAX_FILES and len(processed) >= MAX_FILES:
            break
    print(f'Completed training for {len(processed)} files.')

## 4. Run order

Use the notebook from top to bottom. If you only want a smoke test, set `MAX_FILES = 1` in each batch cell.

## 5. Archive outputs

Create one timestamped archive in Drive containing trained models, reports, and artifacts.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import zipfile

ARCHIVE_DIR = BASE / 'archives'
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
archive_path = ARCHIVE_DIR / f'tradingbot26_outputs_{timestamp}.zip'

include_dirs = [MODEL_DIR, REPORT_DIR, ARTIFACT_DIR]
with zipfile.ZipFile(archive_path, mode='w', compression=zipfile.ZIP_DEFLATED) as archive:
    for directory in include_dirs:
        if not directory.exists():
            print(f'Skipping missing directory: {directory}')
            continue
        for path in directory.rglob('*'):
            if path.is_file():
                archive.write(path, arcname=path.relative_to(BASE))

size_mb = archive_path.stat().st_size / (1024 * 1024)
print(f'Archive written: {archive_path}')
print(f'Size: {size_mb:.2f} MB')